# Asistente experto en analitica de futbol profesional con Gemini, RAG y LangGraph

Proyecto final de IA Generativa: construccion de un agente experto capaz de responder preguntas sobre analitica de futbol profesional usando una base de conocimiento vectorial propia, Gemini como LLM, Gemini Embeddings, ChromaDB, LangGraph y memoria conversacional.

**Dominio elegido:** scouting y analitica avanzada de jugadores de futbol profesional en las cinco grandes ligas europeas durante la temporada 2024-2025.

## Requisitos del enunciado

Este notebook se construye para cumplir los puntos obligatorios del proyecto:

- Base de conocimiento vectorial en ChromaDB usando Gemini Embeddings.
- Minimo de 3 documentos o equivalente a unas 20 paginas de texto, usando CSV y documento propio.
- Pipeline RAG: ingesta, chunking/preprocesamiento, embeddings, almacenamiento, recuperacion y generacion.
- Agente con LangGraph que combine RAG, Gemini y memoria conversacional.
- System prompt personalizado y justificado.
- Celda de interaccion por texto dentro del notebook.
- Minimo de 5 preguntas de ejemplo con respuestas documentadas.

## 1. Instalacion de dependencias

Ejecuta esta celda solo si tu entorno no tiene instaladas las librerias. En local se recomienda usar `requirements.txt`; en Google Colab puedes descomentar la instalacion.

In [4]:
# Si estas en Google Colab o en un entorno nuevo, descomenta y ejecuta:
# !pip install -q langchain langchain-community langchain-google-genai langgraph chromadb python-dotenv pypdf pandas

## 2. Imports y configuracion del entorno

Las claves y configuraciones sensibles se cargan desde `.env`. No se debe hardcodear la API key en el notebook.

In [5]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "models/text-embedding-004")
CHROMA_DB_DIR = os.getenv("CHROMA_DB_DIR", "chroma_db")

if not GOOGLE_API_KEY:
    raise ValueError("No se encontro GOOGLE_API_KEY. Revisa el archivo .env")

print("API key cargada:", bool(GOOGLE_API_KEY))
print("Modelo Gemini:", GEMINI_MODEL)
print("Modelo embeddings:", GEMINI_EMBEDDING_MODEL)
print("Directorio Chroma:", CHROMA_DB_DIR)

c:\Users\User\miniconda3\envs\master_ds_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


API key cargada: True
Modelo Gemini: gemini-2.5-flash
Modelo embeddings: models/text-embedding-004
Directorio Chroma: chroma_db


## 3. Rutas del proyecto

Centralizamos las rutas para que el notebook sea mas mantenible y pueda ejecutarse desde la carpeta `notebooks` o desde la raiz del proyecto.

In [6]:
NOTEBOOK_DIR = Path.cwd()

# Si el notebook se ejecuta desde /notebooks, la raiz es el directorio padre.
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data"
CSV_PATH = DATA_DIR / "players_data-2024_2025.csv"
GLOSSARY_PATH = DATA_DIR / "glosario_metricas_futbol.md"
CHROMA_PATH = PROJECT_ROOT / CHROMA_DB_DIR

required_paths = [CSV_PATH, GLOSSARY_PATH]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"No se encontro el archivo requerido: {path}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV:", CSV_PATH)
print("Glosario:", GLOSSARY_PATH)
print("ChromaDB:", CHROMA_PATH)

PROJECT_ROOT: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve
CSV: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\data\players_data-2024_2025.csv
Glosario: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\data\glosario_metricas_futbol.md
ChromaDB: c:\Users\User\Documents\Master Data Science_IA\IA_GENERATIVA\Asistente_Experto_LLM_Oscar_Fernandez\Asistente_Experto_LLM_Oscar_Fernandez_Evolve\chroma_db


## 4. Inicializacion de Gemini

Creamos dos componentes: el modelo conversacional para generar respuestas y el modelo de embeddings para indexar y recuperar informacion desde ChromaDB.

In [11]:
llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0.5,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model=GEMINI_EMBEDDING_MODEL,
    google_api_key=GOOGLE_API_KEY,
)

print("LLM inicializado:", GEMINI_MODEL)
print("Embeddings inicializados:", GEMINI_EMBEDDING_MODEL)

LLM inicializado: gemini-2.5-flash
Embeddings inicializados: models/text-embedding-004


## 5. Prueba minima del LLM

Antes de construir el RAG, verificamos que Gemini responde correctamente. Esta prueba no usa todavia la base de conocimiento; solo valida conexion y configuracion.

In [12]:
respuesta = llm.invoke("Explica en una frase que significa xG en futbol.")
print(respuesta.content)

xG (Expected Goals) es una métrica que mide la probabilidad de que un disparo se convierta en gol, basándose en la calidad de la ocasión y las condiciones en que se realizó el tiro.


## 6. Carga inicial del dataset y del glosario

El CSV aporta datos estadisticos de jugadores y el glosario aporta conocimiento textual para interpretar las metricas. Ambos se usaran como base de conocimiento del RAG.

In [13]:
df = pd.read_csv(CSV_PATH)
glosario_texto = GLOSSARY_PATH.read_text(encoding="utf-8")

print("Dataset:", df.shape)
print("Columnas:", len(df.columns))
print("Longitud del glosario:", len(glosario_texto), "caracteres")

df[["Player", "Squad", "Comp", "Pos", "Age", "Min", "xG", "xAG", "PrgC", "PrgP"]].head()

Dataset: (2854, 267)
Columnas: 267
Longitud del glosario: 18177 caracteres


,Player,Squad,Comp,Pos,Age,Min,xG,xAG,PrgC,PrgP
0,Max Aarons,Bournemouth,eng Premier League,DF,24.0,86,0.0,0.0,1,8
1,Max Aarons,Valencia,es La Liga,"DF,MF",24.0,120,0.0,0.0,0,6
2,Rodrigo Abajas,Valencia,es La Liga,DF,21.0,65,0.1,0.0,3,2
3,James Abankwah,Udinese,it Serie A,"DF,MF",20.0,88,0.1,0.0,3,4
4,Keyliane Abdallah,Marseille,fr Ligue 1,FW,18.0,3,0.0,0.0,1,0


## 7. Revision rapida de cobertura de datos

Comprobamos que el dataset tiene suficiente volumen y variedad para justificar el proyecto: varias ligas, posiciones y miles de registros.

In [14]:
print("Registros:", len(df))
print("Ligas:")
display(df["Comp"].value_counts())

print("Posiciones principales:")
display(df["Pos"].value_counts().head(15))

Registros: 2854
Ligas:


Comp
it Serie A            634
es La Liga            601
eng Premier League    574
fr Ligue 1            553
de Bundesliga         492
Name: count, dtype: int64

Posiciones principales:


Pos
DF       859
MF       589
FW       371
FW,MF    325
MF,FW    230
GK       212
DF,MF    110
MF,DF     81
DF,FW     53
FW,DF     24
Name: count, dtype: int64

## 8. Siguiente paso

A partir de aqui construiremos la base vectorial:

1. Convertir filas del CSV en documentos textuales de scouting.
2. Dividir el glosario en chunks.
3. Crear embeddings con Gemini.
4. Guardar todo en ChromaDB.
5. Probar el retriever antes de conectar LangGraph.